# 05 - Model zona kedaluwarsa 4 KELAS

Melatih YOLOv8 untuk membedakan **`date` / `code` / `due` / `prod`**, menggantikan
model 1 kelas (`expired_date_zone`) yang terpasang sekarang.

**Kenapa:** jawaban salah `chees` berasal dari parser yang membaca **nomor produksi**
`210526` sebagai tanggal. Memperbaikinya dari sisi parser sudah dicoba 20-11 dan
**gagal** - teks OCR-nya tidak memuat label "Kode Produksi" sama sekali. Dengan model
4 kelas, wilayah kode produksi ditolak **sebelum** OCR, jadi labelnya tidak perlu terbaca.

**Harapan yang jujur:** batas atas perbaikannya sekitar **+6 poin** (1 berkas dari 16).
Jalur zona sudah terukur sebagai optimasi **latensi**, bukan akurasi - 81% sama dengan
maupun tanpa jalur zona (02-10). Jangan harapkan lompatan.

Aplikasi **sudah siap** menerima model ini: `pick_zone` memakai
`expiry.zone_classes_date` dan `zone_classes_reject`, dan mundur aman ke perilaku lama
selama kelasnya kosong. Jadi memasang bobot baru tidak butuh perubahan kode.


In [ ]:
# Sel 1 - PENJAGA GPU. Latihan k-fold uang v3 pernah diam-diam jalan di CPU
# (1 epoch = 965 detik, ~80 jam untuk 5 lipatan) karena !nvidia-smi hanya MENCETAK.
import torch
assert torch.cuda.is_available(), (
    'Runtime ini TANPA GPU. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Sel 2 - pustaka
%pip -q install ultralytics roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)


In [ ]:
# Sel 3 - konfigurasi.
#
# Kunci dibaca dari Colab Secrets (ikon kunci di bilah kiri), BUKAN ditulis di sel.
# Dua alasan: notebook ini ikut di-commit, dan kunci yang tertulis di sel akan
# ikut tersimpan ke Git tanpa ada yang sadar.
#
# Nama secret: ROBOFLOW_API_KEY, dan saklar 'Notebook access' harus MENYALA.
#
# WAJIB kunci PRIVATE (app.roboflow.com > Settings > API Keys > Private API Key).
# Publishable key yang berawalan rf_ akan ditolak 401 'This API key does not exist'.
PESAN = (
    'Secret ROBOFLOW_API_KEY tidak terbaca. Buka ikon kunci di bilah kiri, '
    'buat secret bernama persis ROBOFLOW_API_KEY, isi dengan Private API Key, '
    'lalu nyalakan saklar Notebook access.')
try:
    from google.colab import userdata
    RF_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception as e:
    raise SystemExit(f'{PESAN} Pesan aslinya: {e}')

WORKSPACE    = 'esigelec-rniez'   # 700 gambar, 4 kelas, CC BY 4.0,
                                  # split train/valid/test lengkap
PROJECT      = 'expiry'
VERSI        = 1          # diperiksa 20-11: versi 1 satu-satunya yang ada

EPOCHS       = 80
IMGSZ        = 640         # samakan dengan vision.expiry.imgsz di settings.yaml
RUN_NAME     = 'zone_4kelas_v1'

# Kelas yang WAJIB ada. Kalau dataset ternyata lain, sel 5 berhenti keras -
# lebih baik gagal nyaring daripada melatih model yang kelasnya tidak cocok
# dengan config aplikasi.
KELAS_WAJIB  = {'date', 'code', 'due', 'prod'}
assert RF_API_KEY, PESAN
print('kunci terbaca,', len(RF_API_KEY), 'karakter')


In [ ]:
# Sel 4 - unduh dataset
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
proyek = rf.workspace(WORKSPACE).project(PROJECT)
versi = proyek.version(VERSI)
ds = versi.download('yolov8')
print('diunduh ke:', ds.location)


In [ ]:
# Sel 5 - PERIKSA DATASET SEBELUM MELATIH.
# Dua jebakan yang sudah pernah merusak latihan di proyek ini:
#   1. nama kelas tidak sesuai harapan (dataset Roboflow berlabel tertukar, 27-09)
#   2. gambar TANPA label di split train - dalam YOLO itu berarti 'tidak ada objek
#      di gambar ini', dan itu merusak k-fold uang pertama (07-10)
import collections
from pathlib import Path
import yaml

data_yaml = Path(ds.location) / 'data.yaml'
d = yaml.safe_load(data_yaml.read_text())
nama = {i: n for i, n in enumerate(d['names'])}
print('kelas:', nama)
kurang = KELAS_WAJIB - {n.lower() for n in nama.values()}
assert not kurang, f'Kelas yang diharapkan tidak ada: {kurang}. Periksa datasetnya.'

for split in ('train', 'valid', 'test'):
    img = Path(ds.location, split, 'images')
    lab = Path(ds.location, split, 'labels')
    if not img.is_dir():
        print(f'{split:6} TIDAK ADA')
        continue
    gambar = sorted(p for p in img.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    kosong, hitung = 0, collections.Counter()
    for g in gambar:
        t = lab / (g.stem + '.txt')
        baris = [b for b in t.read_text().splitlines() if b.strip()] if t.exists() else []
        if not baris:
            kosong += 1
        for b in baris:
            hitung[nama.get(int(b.split()[0]), '?')] += 1
    print(f'{split:6} {len(gambar):5} gambar, {kosong:4} TANPA label, kotak: {dict(hitung)}')

print('Kalau jumlah TANPA label di train besar, HENTIKAN dan periksa dulu.')


In [ ]:
# Sel 6 - di mana hasil latihan akan disimpan?
#
# JANGAN beri `project` relatif ke model.train(). Di Colab
# settings['runs_dir'] sudah bernilai /content/runs/detect, dan `project`
# relatif DITAMBAHKAN di belakangnya - jadi project='runs/detect' menghasilkan
#   /content/runs/detect/runs/detect/<nama>/weights/best.pt
# Terbukti 20-11 di Colab sungguhan, dan itu membuat Sel 8 gagal
# FileNotFoundError padahal latihannya berhasil.
from pathlib import Path
from ultralytics import settings as yolo_settings
runs_dir = Path(yolo_settings['runs_dir'])
print('runs_dir ultralytics :', runs_dir)
calon = runs_dir / 'detect' / RUN_NAME
print('perkiraan save_dir   :', calon)
if (calon / 'weights' / 'best.pt').exists():
    raise SystemExit(
        f'{calon} sudah berisi bobot. Ganti RUN_NAME, atau hapus foldernya dengan sadar. '
        'Ultralytics tidak menimpa - ia menaikkan nama diam-diam, dan kamu akan bingung '
        'folder mana yang baru.')


In [ ]:
# Sel 7 - latih.
#
# Tanpa `project`: biarkan ultralytics memakai runs_dir bawaannya (lihat Sel 6).
# `hasil_dir` diambil dari save_dir yang DILAPORKAN trainer, bukan disusun tangan -
# jalur tebakan sudah terbukti meleset dua kali.
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ,
            name=RUN_NAME, seed=0, deterministic=True)

hasil_dir = Path(model.trainer.save_dir)
bobot = hasil_dir / 'weights' / 'best.pt'
print('save_dir sesungguhnya :', hasil_dir)
print('bobot ada             :', bobot.exists())
assert bobot.exists(), f'Latihan selesai tetapi {bobot} tidak ada.'


In [ ]:
# Sel 8 - evaluasi pada split TEST, per kelas
terbaik = YOLO(str(bobot))
m = terbaik.val(data=str(data_yaml), split='test', imgsz=IMGSZ)
print(f'mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}')
for i, n in nama.items():
    try:
        print(f'  {n:6} AP50 {m.box.ap50[i]:.3f}')
    except (IndexError, TypeError):
        pass
print('Yang paling menentukan untuk aplikasi: AP kelas `date` dan `code`.')
print('`code` penting karena gunanya MENOLAK, bukan membaca.')


In [ ]:
# Sel 9 - ukuran bobot = penanda latihan selesai normal
# Ultralytics membuang state optimizer HANYA saat latihan berakhir normal.
# ~6 MB = selesai; ~32 MB = terputus (pelajaran k-fold uang v3, 09-11).
mb = bobot.stat().st_size / 1e6
print(f'{bobot} = {mb:.1f} MB ->', 'selesai normal' if mb < 15 else 'TERPUTUS, jangan dipakai')
print()
print('Unduh berkas itu, lalu salin ke models/best_zone_4kelas.pt di repo.')
print('JANGAN menimpa models/best_zone.pt - itu bukti angka 81% yang sekarang.')


## Memasang bobotnya

1. Unduh `runs/detect/zone_4kelas_v1/weights/best.pt`
2. Salin ke `models/best_zone_4kelas.pt` di repo (**jangan timpa** `best_zone.pt`,
   itu bukti angka 81% yang sekarang)
3. Di `config/settings.yaml` ubah `expiry.zone_weights` ke bobot baru
4. **Ukur sebelum menyimpulkan:**

```
python scripts\ukur_kedaluwarsa.py
```

Yang dibandingkan **benar / salah / diam**, bukan satu angka akurasi - arah kesalahan
di modul ini sudah tiga kali terbukti lebih penting daripada jumlahnya. Dasar
pembanding sekarang: **13 benar / 2 salah / 1 diam = 81%**.

Kalau `chees` berubah dari SALAH menjadi BENAR atau DIAM, perbaikannya nyata.
Kalau tidak ada yang berubah, bobot 4 kelas **tidak membeli apa pun** untuk data ini
dan itu harus dicatat apa adanya, bukan disimpan diam-diam.
